# E05 Credit Card Fraud：时间划分、极端不均衡与 PR-AUC

> Issue #9 / Epic #6 / 路线图 E05

## 你将学到
1. 正类极稀 → Accuracy 几乎无意义
2. 应按 `Time` 切分，避免用未来信息训练
3. 关注 PR-AUC / Recall@Precision，而不是 ROC 一家独大

## 数据
- 需要本地下载 Kaggle `creditcard.csv` → `data/creditcard.csv`
- 若缺失，本 Notebook 会用合成不均衡数据演示流程（标注为 DEMO）


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载真实数据或合成 DEMO


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

path = Path('../../data/creditcard.csv')
DEMO = False
if path.exists():
    df = pd.read_csv(path)
    if QUICK and len(df) > 80000:
        # 保持正类：分层抽样
        pos = df[df['Class'] == 1]
        neg = df[df['Class'] == 0].sample(min(20000, (df['Class'] == 0).sum()), random_state=RANDOM_STATE)
        df = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_STATE)
    print('loaded', path, df.shape)
    print(df['Class'].value_counts(normalize=True))
else:
    DEMO = True
    print('未找到 creditcard.csv，使用合成不均衡 DEMO 数据')
    X, y = make_classification(
        n_samples=8000 if QUICK else 30000,
        n_features=20, n_informative=8, weights=[0.985, 0.015],
        flip_y=0.01, random_state=RANDOM_STATE,
    )
    df = pd.DataFrame(X, columns=[f'V{i}' for i in range(X.shape[1])])
    df['Time'] = np.arange(len(df))
    df['Amount'] = np.abs(np.random.RandomState(RANDOM_STATE).normal(50, 30, len(df)))
    df['Class'] = y
    print(df['Class'].value_counts(normalize=True))
df.head()


## 2. EDA：类别比例与 Amount


In [ ]:
from wanglaoshi import DataAnalyzer

eda_df = df.sample(min(len(df), 5000), random_state=RANDOM_STATE) if QUICK else df
analyzer = DataAnalyzer(eda_df)
display(analyzer.basic_statistics().head(10))
print('fraud rate', df['Class'].mean())
print(df.groupby('Class')['Amount'].describe() if 'Amount' in df.columns else 'no Amount')


## 3. 时间切分（禁止随机打乱）


In [ ]:
# 按 Time 排序后切末尾 20% 作验证
work = df.sort_values('Time').reset_index(drop=True)
split = int(len(work) * 0.8)
train, valid = work.iloc[:split], work.iloc[split:]
feature_cols = [c for c in work.columns if c not in ('Class', 'Time')]
Xtr, Xva = train[feature_cols], valid[feature_cols]
ytr, yva = train['Class'], valid['Class']
print('train fraud rate', ytr.mean(), 'valid fraud rate', yva.mean())
print('time range train', train['Time'].min(), '->', train['Time'].max())
print('time range valid', valid['Time'].min(), '->', valid['Time'].max())


## 4. 基线 + PR-AUC（忽略 Accuracy）


In [ ]:
from wanglaoshi import CompetitionTools as CT
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xtr_s = pd.DataFrame(scaler.fit_transform(Xtr), columns=Xtr.columns)
Xva_s = pd.DataFrame(scaler.transform(Xva), columns=Xva.columns)

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_s, ytr, Xva_s, yva, task_type='classification')
comparison = baseline.compare_models(results, top_n=5)
display(comparison)

best = comparison.iloc[0]['Model']
model = results[best]['model']
# 尽量取概率
if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(Xva_s)[:, 1]
else:
    proba = results[best]['predictions']
pred = (np.asarray(proba) >= 0.5).astype(int) if np.ndim(proba) else results[best]['predictions']
print(classification_report(yva, pred, zero_division=0))
try:
    print('PR-AUC', average_precision_score(yva, proba))
    print('ROC-AUC', roc_auc_score(yva, proba))
except Exception as e:
    print('metric skip', e)


## 5. 阈值：在固定 Precision 下最大化 Recall


In [ ]:
from sklearn.metrics import precision_recall_curve

if hasattr(model, 'predict_proba'):
    prec, rec, ths = precision_recall_curve(yva, proba)
    # 找 precision >= 0.5 时最高 recall
    mask = prec[:-1] >= 0.5
    if mask.any():
        idx = np.argmax(rec[:-1][mask])
        thr = ths[mask][idx]
        print('chosen thr', thr, 'precision', prec[:-1][mask][idx], 'recall', rec[:-1][mask][idx])
    else:
        print('无 precision>=0.5 的阈值，请放宽约束')
else:
    print('模型无 predict_proba，跳过阈值搜索')


## 6. 模拟提交


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

out = Path('outputs/05_credit_card_fraud'); out.mkdir(parents=True, exist_ok=True)
pred_label = results[best]['predictions']
sample = pd.DataFrame({'Id': np.arange(len(pred_label)), 'Class': yva.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred_label, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
print('DEMO mode =', DEMO)


## 7. 检查清单
- [x] 时间切分而非随机划分
- [x] 报告 PR-AUC
- [x] 无数据时 DEMO 可跑通流程
- [ ] class_weight / 重采样对比
- [ ] 成本敏感阈值上线验证
